# Split large prediction CSVs

The combined notebooks (`04b`, `05d`) write a single `predictions.csv` that is too large for
GitHub's 100 MB per-file limit. This notebook splits each oversized `predictions.csv` into
row-chunks of <= `MAX_PART_MB` under a `parts/` subfolder (header kept in every part), so the
data can be committed. The full `predictions.csv` itself is git-ignored.

Run this **after** re-running `04b`/`05d`. Use the last cell to reassemble the full file.

In [1]:
from pathlib import Path
import math
import pandas as pd

def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for p in [start, *start.parents]:
        if (p / 'src').exists() and (p / 'requirements.txt').exists():
            return p
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root()
PRED = PROJECT_ROOT / 'src' / 'data' / 'predictions'
MAX_PART_MB = 45  # keep each part well under GitHub's 100 MB limit (and the 50 MB warning)

TARGETS = [
    PRED / 'dataset_1' / 'centrality_threshold' / 'predictions.csv',
    PRED / 'dataset_1' / 'embedding_threshold' / 'predictions.csv',
]

## Split each file into `parts/`

In [2]:
def split_csv(path, max_part_mb=MAX_PART_MB):
    if not path.exists():
        print(f'skip (missing): {path.relative_to(PROJECT_ROOT)}')
        return
    size_mb = path.stat().st_size / 1e6
    n_parts = max(1, math.ceil(size_mb / max_part_mb))
    df = pd.read_csv(path)
    parts_dir = path.parent / 'predictions_parts'
    parts_dir.mkdir(exist_ok=True)
    for old in parts_dir.glob('predictions_part_*.csv'):
        old.unlink()
    rows_per = math.ceil(len(df) / n_parts)
    for i in range(n_parts):
        chunk = df.iloc[i * rows_per:(i + 1) * rows_per]
        if chunk.empty:
            continue
        out = parts_dir / f'predictions_part_{i + 1:02d}.csv'
        chunk.to_csv(out, index=False)
        print(f'  {out.relative_to(PROJECT_ROOT)}  ({out.stat().st_size/1e6:.1f} MB, {len(chunk)} rows)')
    print(f'{path.relative_to(PROJECT_ROOT)}: {size_mb:.0f} MB, {len(df)} rows -> {n_parts} parts')

for t in TARGETS:
    split_csv(t)

  src/data/predictions/combined_centrality_threshold/predictions_parts/predictions_part_01.csv  (41.7 MB, 258977 rows)


  src/data/predictions/combined_centrality_threshold/predictions_parts/predictions_part_02.csv  (41.5 MB, 258977 rows)


  src/data/predictions/combined_centrality_threshold/predictions_parts/predictions_part_03.csv  (41.4 MB, 258977 rows)


  src/data/predictions/combined_centrality_threshold/predictions_parts/predictions_part_04.csv  (41.5 MB, 258977 rows)


  src/data/predictions/combined_centrality_threshold/predictions_parts/predictions_part_05.csv  (40.9 MB, 258976 rows)
src/data/predictions/combined_centrality_threshold/predictions.csv: 210 MB, 1294884 rows -> 5 parts


  src/data/predictions/combined_embedding_threshold/predictions_parts/predictions_part_01.csv  (44.7 MB, 323721 rows)


  src/data/predictions/combined_embedding_threshold/predictions_parts/predictions_part_02.csv  (44.0 MB, 323721 rows)


  src/data/predictions/combined_embedding_threshold/predictions_parts/predictions_part_03.csv  (44.1 MB, 323721 rows)


  src/data/predictions/combined_embedding_threshold/predictions_parts/predictions_part_04.csv  (43.4 MB, 323721 rows)
src/data/predictions/combined_embedding_threshold/predictions.csv: 179 MB, 1294884 rows -> 4 parts


## Reassemble (optional) — rebuild the full `predictions.csv` from its parts

In [3]:
def reassemble(pred_dir):
    parts = sorted((pred_dir / 'predictions_parts').glob('predictions_part_*.csv'))
    if not parts:
        print(f'no parts in {pred_dir.relative_to(PROJECT_ROOT)}'); return
    df = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
    out = pred_dir / 'predictions.csv'
    df.to_csv(out, index=False)
    print(f'{out.relative_to(PROJECT_ROOT)}: {len(df)} rows from {len(parts)} parts')

# Uncomment to rebuild:
# for t in TARGETS:
#     reassemble(t.parent)